# Hooks y Steering: controlá lo que hace tu agente

**Student Builder Group · Universidad Austral · 20/11/2026**

Un agente decide solo qué tools llamar. Eso es lo poderoso... y lo peligroso. Strands te da tres formas de meterte en el **agent loop** sin tocar la lógica del agente:

```
pedido ──> [ modelo piensa ] ──> [ llama a una tool ] ──> [ modelo piensa ] ──> respuesta
                  ▲                       ▲
                  │                       │
          acá podés observar, bloquear o corregir
```

| Herramienta | ¿Qué hace? | ¿Quién decide? |
|---|---|---|
| **Hooks** | Observar y reaccionar (logs, métricas, auditoría) | Tu código |
| **Interventions** | Permitir o bloquear acciones (reglas, permisos) | Tu código |
| **Steering** | Corregir al agente y que vuelva a intentar | Reglas o **otro LLM** |

> Igual que en el notebook de multi-agente: tools con **datos inventados**, sin APIs externas. Solo necesitás acceso a Bedrock.

---
## 0. Setup

In [ ]:
%pip install -q strands-agents

In [ ]:
import os
import boto3
from strands import Agent, tool
from strands.models.bedrock import BedrockModel

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"  # rápido y barato; podés cambiarlo por uno más grande
REGION = "us-east-1"
AWS_PROFILE = None  # None = perfil por defecto; o poné el nombre de tu perfil, ej. "mi-perfil"

if AWS_PROFILE:
    os.environ["AWS_PROFILE"] = AWS_PROFILE
SESION = boto3.Session(profile_name=AWS_PROFILE, region_name=REGION)


def modelo():
    return BedrockModel(model_id=MODEL_ID, boto_session=SESION)


# Dos tools de mentira: buscar vuelos y reservar (¡reservar es irreversible!)
VUELOS = {
    "bariloche": [{"aerolinea": "Flybondi", "precio_usd": 85}, {"aerolinea": "Aerolíneas Argentinas", "precio_usd": 120}],
    "ushuaia": [{"aerolinea": "Aerolíneas Argentinas", "precio_usd": 210}],
}


@tool
def buscar_vuelos(destino: str) -> list:
    '''Busca vuelos desde Buenos Aires hacia un destino de Argentina.

    Args:
        destino: ciudad de destino, por ejemplo "Bariloche".
    '''
    return VUELOS.get(destino.lower(), [])


@tool
def reservar_vuelo(destino: str, aerolinea: str) -> dict:
    '''Reserva un vuelo y cobra la tarjeta en el momento.

    Args:
        destino: ciudad de destino.
        aerolinea: aerolínea elegida.
    '''
    return {"estado": "reservado", "codigo": "SBG-001", "destino": destino, "aerolinea": aerolinea}


print("Setup OK")

---
## 1. Hooks: observar el agent loop

Un **hook** es una función que Strands llama en un momento del loop. Hay eventos para todo: `BeforeInvocationEvent`, `BeforeModelCallEvent`, `BeforeToolCallEvent`, `AfterToolCallEvent`, etc.

Lo más simple: una función con el **tipo del evento** en la firma. Strands mira el type hint y sabe cuándo llamarla.

In [ ]:
from strands.hooks import BeforeToolCallEvent, AfterToolCallEvent


def antes_de_la_tool(event: BeforeToolCallEvent):
    print(f"🔧 {event.tool_use['name']}({event.tool_use['input']})")


def despues_de_la_tool(event: AfterToolCallEvent):
    if event.cancel_message:  # alguien (intervention o steering) la canceló
        print(f"   ↳ ❌ cancelada: {event.cancel_message}")
    else:
        print(f"   ↳ ✅ ok en {event.duration * 1000:.1f} ms")


agente = Agent(
    model=modelo(),
    tools=[buscar_vuelos],
    hooks=[antes_de_la_tool, despues_de_la_tool],  # ← así de simple
    system_prompt="Sos un asistente de viajes para estudiantes. Respondé corto en español.",
    callback_handler=None,
)

print("\n" + str(agente("¿Qué vuelos hay a Bariloche?")))

Si querés **agrupar** varios hooks en algo reutilizable (por ejemplo, para compartirlo entre agentes), usás una clase `HookProvider`:

In [ ]:
from strands.hooks import HookProvider, HookRegistry


class Observabilidad(HookProvider):
    def register_hooks(self, registry: HookRegistry, **kwargs):
        registry.add_callback(BeforeToolCallEvent, antes_de_la_tool)
        registry.add_callback(AfterToolCallEvent, despues_de_la_tool)


agente = Agent(model=modelo(), tools=[buscar_vuelos], hooks=[Observabilidad()], callback_handler=None)
print("\n" + str(agente("¿Y a Ushuaia?")))

---
## 2. Interventions: bloquear lo que no está permitido

Para **permitir o bloquear** acciones, Strands tiene `InterventionHandler`. Devolvés una decisión:
- `Proceed()` → sigue normal.
- `Deny(reason=...)` → bloquea la tool; el modelo ve el motivo y se lo explica al usuario.

Ejemplo: una política de la facultad que no permite viajar a Ushuaia (temporada cerrada).

In [ ]:
from strands import InterventionHandler
from strands.interventions import Deny, Proceed

DESTINOS_RESTRINGIDOS = {"ushuaia"}


class PoliticaDeViajes(InterventionHandler):
    name = "politica-de-viajes"

    def before_tool_call(self, event, **kwargs):
        destino = event.tool_use["input"].get("destino", "").lower()
        if destino in DESTINOS_RESTRINGIDOS:
            return Deny(reason=f"Política: no se permiten viajes a {destino} (temporada cerrada).")
        return Proceed()


agente = Agent(
    model=modelo(),
    tools=[buscar_vuelos, reservar_vuelo],
    hooks=[antes_de_la_tool, despues_de_la_tool],  # reusamos los hooks para ver qué pasa
    interventions=[PoliticaDeViajes()],
    system_prompt="Sos un asistente de viajes para estudiantes. Respondé corto en español.",
    callback_handler=None,
)

print("\n" + str(agente("Reservame el vuelo más barato a Ushuaia.")))

---
## 3. Steering: corregir al agente (sin hardcodear)

El **steering** no solo bloquea: le da **feedback** al agente para que corrija lo que iba a hacer y vuelva a intentar. Las decisiones posibles son:
- `Proceed` → sigue.
- `Guide(reason=...)` → cancela esa acción y le explica al agente qué hacer en su lugar.
- `Interrupt` → frena y le pide input a un humano.

### 3.1 Steering con reglas

Regla: **nunca reservar sin que el usuario confirme**. Si el agente intenta reservar y el usuario todavía no dijo "confirmo", lo guiamos para que primero pregunte.

(Le pedimos al agente que sea "ansioso" y haga todo sin preguntar, para que se vea el steering en acción.)

In [ ]:
from strands.vended_plugins.steering import SteeringHandler, Guide, Proceed as SteeringProceed


def ultimo_mensaje_del_usuario(agent):
    for msg in reversed(agent.messages):
        textos = [c["text"] for c in msg["content"] if "text" in c]
        if msg["role"] == "user" and textos:
            return " ".join(textos).lower()
    return ""


class ConfirmarAntesDeReservar(SteeringHandler):
    async def steer_before_tool(self, *, agent, tool_use, **kwargs):
        if tool_use["name"] == "reservar_vuelo" and "confirmo" not in ultimo_mensaje_del_usuario(agent):
            return Guide(reason="Antes de reservar, mostrale al usuario aerolínea y precio y pedile que responda 'confirmo'.")
        return SteeringProceed(reason="ok")


ANSIOSO = "Sos un asistente de viajes. Hacé lo que te pide el usuario directamente, sin hacer preguntas. Respondé corto en español."

agente = Agent(
    model=modelo(),
    tools=[buscar_vuelos, reservar_vuelo],
    hooks=[antes_de_la_tool, despues_de_la_tool],  # reusamos los hooks para ver qué pasa
    plugins=[ConfirmarAntesDeReservar()],  # el steering es un plugin
    system_prompt=ANSIOSO,
    callback_handler=None,
)

print("\n" + str(agente("Reservame el vuelo más barato a Bariloche.")))

In [ ]:
# El agente recuerda la conversación: ahora sí confirmamos
print(str(agente("Sí, confirmo Flybondi por 85 USD.")))

### 3.2 Steering con un LLM

Escribir reglas para todo es imposible. Con `LLMSteeringHandler`, **otro LLM** evalúa cada tool call contra una política escrita en lenguaje natural y decide si sigue, guía o interrumpe.

Ojo: el juez no ve la conversación, ve un **registro (ledger) de las tool calls** que hizo el agente. Por eso la política habla de tool calls.

In [ ]:
from strands.vended_plugins.steering import LLMSteeringHandler

juez = LLMSteeringHandler(
    model=modelo(),
    system_prompt=(
        "Política: reservar_vuelo solo se puede llamar si antes hubo un buscar_vuelos exitoso "
        "para el mismo destino. Si no lo hubo, guiá al agente para que primero busque vuelos. "
        "Escribí el motivo en español."
    ),
)

agente = Agent(
    model=modelo(),
    tools=[buscar_vuelos, reservar_vuelo],
    hooks=[antes_de_la_tool, despues_de_la_tool],  # reusamos los hooks para ver qué pasa
    plugins=[juez],
    system_prompt=ANSIOSO,
    callback_handler=None,
)

# El agente "ansioso" intenta reservar directo; el juez lo frena y lo manda a buscar primero
print("\n" + str(agente("Reservame Aerolíneas Argentinas a Bariloche.")))

---
## Resumen: ¿cuál uso?

| Si querés... | Usá | Ejemplo |
|---|---|---|
| Ver qué hace el agente (logs, tiempos, auditoría) | **Hooks** | Loggear cada tool call |
| Bloquear acciones con una regla clara | **Interventions** | "No se viaja a Ushuaia" |
| Corregir al agente con reglas simples | **Steering** con `SteeringHandler` | "Pedí confirmación antes de reservar" |
| Corregir al agente con criterio (sin escribir cada regla) | **Steering** con `LLMSteeringHandler` | "Acciones irreversibles requieren confirmación" |

### Para practicar 💪
1. Agregá un hook con `BeforeInvocationEvent` que imprima cada pedido del usuario.
2. Hacé que `PoliticaDeViajes` bloquee reservas de más de 150 USD.
3. Cambiá la política del `LLMSteeringHandler` y fijate cómo cambia el comportamiento.